In [1]:
import os
import numpy as np
from PIL import Image

class FeatureExtractor:
    def __init__(self, grid_size=(5, 5)):
        self.grid_size = grid_size

    def extract(self, image_path, binarize_for_hopfield=False):
        img = Image.open(image_path).convert('L')
        binary_img = (np.array(img) < 128).astype(int)
        
        h, w = binary_img.shape
        rows, cols = self.grid_size
        cell_h, cell_w = h // rows, w // cols
        
        vector = []
        for i in range(rows):
            for j in range(cols):
                cell = binary_img[i*cell_h:(i+1)*cell_h, j*cell_w:(j+1)*cell_w]
                vector.append(np.sum(cell))
                
        norm_vector = np.array(vector) / (cell_h * cell_w)
    
        if binarize_for_hopfield:
            return np.where(norm_vector > 0.5, 1, -1)
        return norm_vector

class RosenblattPerceptron:
    def __init__(self, input_size, learning_rate=0.1):
        self.weights = np.random.rand(input_size)
        self.bias = np.random.rand(1)[0]
        self.r = learning_rate

    def predict(self, x):
        return 1 if np.dot(self.weights, x) > self.bias else -1

    def train_step(self, x, d):
        y = self.predict(x)
        if y != d:
            self.weights += self.r * (d - y) * x
        return y == d

class HopfieldNetwork:
    def __init__(self, input_size):
        self.weights = np.zeros((input_size, input_size))

    def train(self, patterns):
        for x in patterns:
            x = x.reshape(-1, 1)
            self.weights += np.dot(x, x.T)
        np.fill_diagonal(self.weights, 0)

    def predict(self, x, max_iter=100):
        y = np.copy(x)
        for _ in range(max_iter):
            y_prev = np.copy(y)
            S = np.dot(self.weights, y)
            y = np.where(S > 0, 1, -1)
            if np.array_equal(y, y_prev):
                return y
        return y

class HammingNetwork:
    def __init__(self, num_classes, input_size):
        self.m = num_classes
        self.n = input_size
        self.w1 = np.zeros((input_size, num_classes))
        self.b = np.zeros(num_classes)
        self.epsilon = 1.0 / (2 * num_classes)

    def train(self, patterns):
        for k in range(self.m):
            self.w1[:, k] = patterns[k] / 2.0
            self.b[k] = self.n / 2.0

    def predict(self, x, max_iter=100):
        y2 = np.dot(x, self.w1) + self.b
        for _ in range(max_iter):
            y2_prev = np.copy(y2)
            y2_new = np.zeros(self.m)
            for j in range(self.m):
                sum_others = sum(y2[k] for k in range(self.m) if k != j)
                S = y2[j] - self.epsilon * sum_others
                y2_new[j] = S if S > 0 else 0
            y2 = y2_new
            if np.allclose(y2, y2_prev):
                break
        return np.argmax(y2)

if __name__ == "__main__":
    extractor = FeatureExtractor(grid_size=(5, 5))
    test_img = "test.bmp"

    print("--- 1. ПЕРЦЕПТРОН РОЗЕНБЛАТТА ---")
    try:
        perceptron = RosenblattPerceptron(input_size=25)
        class_1_files = [f"bmp-shki/1/{f}" for f in os.listdir("bmp-shki/1") if f.endswith(".bmp")]
        class_2_files = [f"bmp-shki/2/{f}" for f in os.listdir("bmp-shki/2") if f.endswith(".bmp")]
        
        for epoch in range(5):
            for path in class_1_files:
                perceptron.train_step(extractor.extract(path), d=1)
            for path in class_2_files:
                perceptron.train_step(extractor.extract(path), d=-1)
                
        p_res = perceptron.predict(extractor.extract(test_img))
        print(f"Перцептрон класа - {'1' if p_res == 1 else '2'}\n")
    except Exception as e:
        print(f"Помилка Перцептрона{e}\n")

    print("--- 2. Мережа ХОПФІЛДА ---")
    try:
        hopfield = HopfieldNetwork(input_size=25)
        h_pat1 = extractor.extract("etalon_1.bmp", binarize_for_hopfield=True)
        h_pat2 = extractor.extract("etalon_2.bmp", binarize_for_hopfield=True)
        h_pat3 = extractor.extract("etalon_3.bmp", binarize_for_hopfield=True)
        
        hopfield.train([h_pat1, h_pat2, h_pat3]) 
        
        h_test = extractor.extract(test_img, binarize_for_hopfield=True)
        h_res_vector = hopfield.predict(h_test)
        
        if np.array_equal(h_res_vector, h_pat1): print("Хопфілд: Клас 1\n")
        elif np.array_equal(h_res_vector, h_pat2): print("Хопфілд: Клас 2\n")
        elif np.array_equal(h_res_vector, h_pat3): print("Хопфілд: Клас 3\n")
        else: print("Хопфілд не впізнав результат\n")
    except Exception as e:
        print(f"Помилка {e}\n")

    print("--- 3. Мережа ХЕМІНГА ---")
    try:
        hamming = HammingNetwork(num_classes=3, input_size=25)
        hm_pat1 = extractor.extract("etalon_1.bmp")
        hm_pat2 = extractor.extract("etalon_2.bmp")
        hm_pat3 = extractor.extract("etalon_3.bmp")
        
        hamming.train([hm_pat1, hm_pat2, hm_pat3]) 
        hm_test = extractor.extract(test_img)
        hm_res = hamming.predict(hm_test) 
        
        print(f"Хемінг: {hm_res + 1}\n")
    except Exception as e:
        print(f"Помилка Хемінга {e}\n")

--- 1. ПЕРЦЕПТРОН РОЗЕНБЛАТТА ---
Перцептрон класа - 2

--- 2. Мережа ХОПФІЛДА ---
Хопфілд: Клас 1

--- 3. Мережа ХЕМІНГА ---
Хемінг: 2

